# Building a Conversational AI Chatbot

Create a streaming chatbot with the OpenAI API and Gradio. This notebook covers conversation history, system prompts, and context-aware responses.

## 1. Setup

Install the required packages if they are not already available:

```bash
pip install openai gradio python-dotenv
```

Add your API key to a `.env` file in the same directory:

```text
OPENAI_API_KEY=your_api_key
OPENAI_MODEL=gpt-4.1-mini
```

In [ ]:
import os

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

SYSTEM_MESSAGE = "You are a helpful assistant."


## 2. Create the chat callback

Gradio passes the latest user message and the conversation history to the callback. Send both to the model so it can respond with the relevant context.

In [ ]:
def chat(message, history):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        *history,
        {"role": "user", "content": message},
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
    )

    return response.choices[0].message.content


## 3. Launch the chatbot

`ChatInterface` provides a simple chat UI and manages the conversation history.

In [ ]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="AI Chatbot",
).launch()


## 4. Stream responses

Streaming displays the answer as it is generated instead of waiting for the complete response.

In [ ]:
def chat(message, history):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        *history,
        {"role": "user", "content": message},
    ]

    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True,
    )

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


## 5. Use a system prompt to guide responses

A system prompt provides the assistant with its role, tone, and business context. This example configures an assistant for a clothing store.

In [ ]:
SYSTEM_MESSAGE = (
    "You are a helpful assistant in a clothing store. "
    "Hats are 60% off and most other items are 50% off. "
    "Gently highlight relevant sale items when helping customers. "
    "If a customer asks about shoes, explain that shoes are not on sale "
    "and remind them about the hats."
)


## 6. Add context based on the user's message

You can add instructions dynamically when a message contains a particular topic. Here, the assistant explains that the store does not sell belts and suggests other sale items.

In [ ]:
def chat(message, history):
    system_message = SYSTEM_MESSAGE

    if "belt" in message.lower():
        system_message += (
            " The store does not sell belts. "
            "If asked about belts, explain this and suggest items that are on sale."
        )

    messages = [
        {"role": "system", "content": system_message},
        *history,
        {"role": "user", "content": message},
    ]

    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True,
    )

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


## 7. Run the final chatbot

Run this cell after defining the final `chat` function.

In [ ]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Clothing Store Assistant",
).launch()


## Key concepts

- **Conversation history:** Previous messages are included so the assistant can follow the conversation.
- **System prompting:** Instructions define the assistant's role, tone, and business rules.
- **Streaming:** Responses are yielded incrementally as the model generates them.
- **Dynamic context:** Additional instructions can be selected based on the user's message.

## Business application

Conversational assistants can support customer service, product discovery, and other business workflows. A system prompt can provide business-specific information and guide the assistant's tone and recommendations.